# Análise de Estoque

## Objetivo

Este notebook analisa a posição atual de estoque da **Distribuidora Horizonte** e combina informações operacionais com a classificação comercial dos produtos.

A análise utiliza:

- estoque disponível;
- estoque mínimo;
- estoque máximo;
- valor financeiro do estoque;
- venda média diária;
- classificação ABC;
- faturamento;
- última venda.

## Indicadores

São calculados:

- dias de cobertura de estoque;
- situação do estoque;
- risco de ruptura;
- excesso de estoque;
- capital imobilizado;
- prioridade de reposição.

## Classificações

Os produtos poderão apresentar situações como:

- Ruptura;
- Estoque crítico;
- Estoque baixo;
- Estoque adequado;
- Excesso de estoque;
- Sem demanda recente.

A classificação ABC é utilizada em conjunto com a situação do estoque para definir a prioridade operacional.

## Saídas

Este notebook cria:

- `analise_estoque_atual`;
- `resumo_estoque`.

## Fluxo

Silver Estoque + Gold Curva ABC → Análise de Estoque → Gold

## Resultado

A posição atual de estoque foi combinada com o comportamento comercial dos produtos.

A análise permitiu identificar:

- produtos em ruptura;
- produtos abaixo do estoque mínimo;
- produtos com poucos dias de cobertura;
- produtos com excesso de estoque;
- produtos sem demanda recente;
- capital potencialmente imobilizado;
- produtos Classe A com risco operacional.

A combinação da Curva ABC com os indicadores de estoque permite priorizar decisões de reposição de acordo com a importância comercial de cada produto.


In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"
schema_gold = "varejo_gold"


print(f"Catálogo: {catalogo_atual}")
print(f"Silver: {schema_silver}")
print(f"Gold: {schema_gold}")

In [0]:
def carregar_silver(nome_tabela):

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


def carregar_gold(nome_tabela):

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )


def salvar_gold(
    dataframe,
    nome_tabela
):

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )

    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )

    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
df_resultado_qualidade = carregar_silver(
    "resultado_qualidade"
)


testes_criticos_reprovados = (

    df_resultado_qualidade

    .filter(
        (F.col("status") == "REPROVADO")
        &
        (F.col("criticidade") == "CRÍTICA")
    )

    .count()
)


if testes_criticos_reprovados > 0:

    raise Exception(
        "Existem testes críticos de qualidade "
        "reprovados. A análise de estoque "
        "não será processada."
    )


print(
    "Quality Gate aprovado."
)

In [0]:
df_estoque = carregar_silver(
    "fato_estoque"
)


df_curva_abc = carregar_gold(
    "produtos_curva_abc"
)

In [0]:
ultima_data_estoque = (

    df_estoque

    .agg(
        F.max(
            "data_referencia"
        ).alias(
            "ultima_data"
        )
    )

    .first()[
        "ultima_data"
    ]
)


print(
    f"Posição de estoque analisada: "
    f"{ultima_data_estoque}"
)

In [0]:
df_estoque_atual = (

    df_estoque

    .filter(
        F.col(
            "data_referencia"
        )
        == F.lit(
            ultima_data_estoque
        )
    )

    .select(
        "data_referencia",
        "id_produto",
        "quantidade_estoque",
        "custo_medio",
        "valor_estoque",
        "estoque_minimo",
        "estoque_maximo"
    )
)


print(
    f"Produtos na posição de estoque: "
    f"{df_estoque_atual.count():,}"
)

In [0]:
df_analise_estoque = (

    df_curva_abc.alias("abc")

    .join(
        df_estoque_atual.alias("e"),
        on="id_produto",
        how="left"
    )

    .select(

        "id_produto",

        F.col(
            "abc.nome_produto"
        ).alias(
            "nome_produto"
        ),

        F.col(
            "abc.categoria"
        ).alias(
            "categoria"
        ),

        F.col(
            "abc.subcategoria"
        ).alias(
            "subcategoria"
        ),

        F.col(
            "abc.marca"
        ).alias(
            "marca"
        ),

        F.col(
            "abc.situacao_produto"
        ).alias(
            "situacao_produto"
        ),

        F.col(
            "abc.classe_abc"
        ).alias(
            "classe_abc"
        ),

        F.col(
            "abc.ranking_faturamento"
        ).alias(
            "ranking_faturamento"
        ),

        F.col(
            "abc.faturamento"
        ).alias(
            "faturamento_12m"
        ),

        F.col(
            "abc.quantidade_vendida"
        ).alias(
            "quantidade_vendida_12m"
        ),

        F.col(
            "abc.venda_media_diaria"
        ).alias(
            "venda_media_diaria"
        ),

        F.col(
            "abc.ultima_venda_periodo"
        ).alias(
            "ultima_venda"
        ),

        F.col(
            "e.data_referencia"
        ).alias(
            "data_referencia"
        ),

        F.col(
            "e.quantidade_estoque"
        ).alias(
            "quantidade_estoque"
        ),

        F.col(
            "e.custo_medio"
        ).alias(
            "custo_medio"
        ),

        F.col(
            "e.valor_estoque"
        ).alias(
            "valor_estoque"
        ),

        F.col(
            "e.estoque_minimo"
        ).alias(
            "estoque_minimo"
        ),

        F.col(
            "e.estoque_maximo"
        ).alias(
            "estoque_maximo"
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "dias_cobertura",

        F.when(
            F.col(
                "venda_media_diaria"
            ) > 0,

            F.round(
                F.col(
                    "quantidade_estoque"
                )
                /
                F.col(
                    "venda_media_diaria"
                ),
                2
            )
        )

        .otherwise(
            F.lit(None)
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "ocupacao_estoque_percentual",

        F.when(
            F.col(
                "estoque_maximo"
            ) > 0,

            F.round(
                F.col(
                    "quantidade_estoque"
                )
                /
                F.col(
                    "estoque_maximo"
                )
                * 100,
                2
            )
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "situacao_estoque",

        # Não existe nenhuma unidade disponível

        F.when(
            F.col(
                "quantidade_estoque"
            ) == 0,

            "Ruptura"
        )


        # Produto sem venda recente,
        # mas com estoque disponível

        .when(
            (
                F.col(
                    "venda_media_diaria"
                ) == 0
            )
            &
            (
                F.col(
                    "quantidade_estoque"
                ) > 0
            ),

            "Sem demanda recente"
        )


        # Estoque abaixo de aproximadamente
        # três dias de demanda

        .when(
            F.col(
                "dias_cobertura"
            ) < 3,

            "Estoque crítico"
        )


        # Estoque abaixo do mínimo definido

        .when(
            F.col(
                "quantidade_estoque"
            )
            <
            F.col(
                "estoque_minimo"
            ),

            "Estoque baixo"
        )


        # Estoque acima do limite recomendado

        .when(
            F.col(
                "quantidade_estoque"
            )
            >
            F.col(
                "estoque_maximo"
            ),

            "Excesso de estoque"
        )


        .otherwise(
            "Estoque adequado"
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "indicador_ruptura",

        F.when(
            F.col(
                "quantidade_estoque"
            ) == 0,
            1
        ).otherwise(0)
    )

    .withColumn(

        "indicador_estoque_baixo",

        F.when(
            F.col(
                "quantidade_estoque"
            )
            <
            F.col(
                "estoque_minimo"
            ),
            1
        ).otherwise(0)
    )

    .withColumn(

        "indicador_excesso",

        F.when(
            F.col(
                "quantidade_estoque"
            )
            >
            F.col(
                "estoque_maximo"
            ),
            1
        ).otherwise(0)
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "prioridade_estoque",

        # Produto essencial para faturamento
        # sem nenhuma unidade disponível

        F.when(
            (
                F.col(
                    "classe_abc"
                ) == "A"
            )
            &
            (
                F.col(
                    "situacao_estoque"
                ) == "Ruptura"
            ),

            "Crítica"
        )


        # Produto A com estoque
        # em situação crítica

        .when(
            (
                F.col(
                    "classe_abc"
                ) == "A"
            )
            &
            (
                F.col(
                    "situacao_estoque"
                ).isin(
                    [
                        "Estoque crítico",
                        "Estoque baixo"
                    ]
                )
            ),

            "Muito alta"
        )


        # Produtos B em ruptura ou nível baixo

        .when(
            (
                F.col(
                    "classe_abc"
                ) == "B"
            )
            &
            (
                F.col(
                    "situacao_estoque"
                ).isin(
                    [
                        "Ruptura",
                        "Estoque crítico",
                        "Estoque baixo"
                    ]
                )
            ),

            "Alta"
        )


        # Qualquer produto com excesso

        .when(
            F.col(
                "situacao_estoque"
            )
            == "Excesso de estoque",

            "Revisar compras"
        )


        # Produto sem venda e ainda com estoque

        .when(
            F.col(
                "situacao_estoque"
            )
            == "Sem demanda recente",

            "Revisar portfólio"
        )


        .otherwise(
            "Normal"
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "acao_recomendada",

        F.when(
            F.col(
                "prioridade_estoque"
            ) == "Crítica",

            "Reposição imediata"
        )

        .when(
            F.col(
                "prioridade_estoque"
            ) == "Muito alta",

            "Priorizar reposição"
        )

        .when(
            F.col(
                "prioridade_estoque"
            ) == "Alta",

            "Programar reposição"
        )

        .when(
            F.col(
                "prioridade_estoque"
            ) == "Revisar compras",

            "Reduzir ou suspender novas compras"
        )

        .when(
            F.col(
                "prioridade_estoque"
            ) == "Revisar portfólio",

            "Avaliar descontinuação ou ação comercial"
        )

        .otherwise(
            "Manter acompanhamento"
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "quantidade_excedente",

        F.when(
            F.col(
                "quantidade_estoque"
            )
            >
            F.col(
                "estoque_maximo"
            ),

            F.col(
                "quantidade_estoque"
            )
            -
            F.col(
                "estoque_maximo"
            )
        )

        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(

        "valor_excedente",

        F.round(
            F.col(
                "quantidade_excedente"
            )
            *
            F.col(
                "custo_medio"
            ),
            2
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(

        "quantidade_abaixo_minimo",

        F.when(
            F.col(
                "quantidade_estoque"
            )
            <
            F.col(
                "estoque_minimo"
            ),

            F.col(
                "estoque_minimo"
            )
            -
            F.col(
                "quantidade_estoque"
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
df_analise_estoque = (

    df_analise_estoque

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_analise_estoque_atual = (

    df_analise_estoque

    .select(

        "data_referencia",

        "id_produto",
        "nome_produto",
        "categoria",
        "subcategoria",
        "marca",
        "situacao_produto",

        "classe_abc",
        "ranking_faturamento",

        "faturamento_12m",
        "quantidade_vendida_12m",
        "venda_media_diaria",
        "ultima_venda",

        "quantidade_estoque",
        "estoque_minimo",
        "estoque_maximo",

        "dias_cobertura",
        "ocupacao_estoque_percentual",

        "custo_medio",
        "valor_estoque",

        "situacao_estoque",
        "prioridade_estoque",
        "acao_recomendada",

        "indicador_ruptura",
        "indicador_estoque_baixo",
        "indicador_excesso",

        "quantidade_abaixo_minimo",
        "quantidade_excedente",
        "valor_excedente",

        "_data_processamento"
    )
)

In [0]:
salvar_gold(
    df_analise_estoque_atual,
    "analise_estoque_atual"
)

In [0]:
display(

    df_analise_estoque_atual

    .groupBy(
        "situacao_estoque"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_produtos"
        ),

        F.round(
            F.sum(
                "valor_estoque"
            ),
            2
        ).alias(
            "valor_estoque"
        )
    )

    .orderBy(
        F.desc(
            "quantidade_produtos"
        )
    )
)

In [0]:
display(

    df_analise_estoque_atual

    .filter(
        (F.col("classe_abc") == "A")
        &
        (F.col("situacao_estoque") == "Ruptura")
    )

    .select(
        "ranking_faturamento",
        "nome_produto",
        "categoria",
        "faturamento_12m",
        "venda_media_diaria",
        "quantidade_estoque",
        "prioridade_estoque",
        "acao_recomendada"
    )

    .orderBy(
        "ranking_faturamento"
    )
)

In [0]:
display(

    df_analise_estoque_atual

    .filter(
        (F.col("classe_abc") == "C")
        &
        (F.col("situacao_estoque") == "Excesso de estoque")
    )

    .select(
        "nome_produto",
        "categoria",
        "faturamento_12m",
        "quantidade_estoque",
        "estoque_maximo",
        "quantidade_excedente",
        "valor_excedente"
    )

    .orderBy(
        F.desc(
            "valor_excedente"
        )
    )

    .limit(30)
)

In [0]:
display(

    df_analise_estoque_atual

    .filter(
        (F.col("classe_abc") == "Sem venda")
        &
        (F.col("quantidade_estoque") > 0)
    )

    .select(
        "id_produto",
        "nome_produto",
        "categoria",
        "quantidade_estoque",
        "valor_estoque",
        "ultima_venda",
        "acao_recomendada"
    )

    .orderBy(
        F.desc(
            "valor_estoque"
        )
    )
)

In [0]:
df_resumo_estoque = (

    df_analise_estoque_atual

    .groupBy(
        "situacao_estoque"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_produtos"
        ),

        F.round(
            F.sum(
                "valor_estoque"
            ),
            2
        ).alias(
            "valor_estoque"
        ),

        F.round(
            F.sum(
                "valor_excedente"
            ),
            2
        ).alias(
            "valor_excedente"
        ),

        F.round(
            F.sum(
                "faturamento_12m"
            ),
            2
        ).alias(
            "faturamento_12m"
        )
    )

    .withColumn(
        "data_referencia",
        F.lit(
            ultima_data_estoque
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_resumo_estoque,
    "resumo_estoque"
)

In [0]:
df_kpis_estoque = (

    df_analise_estoque_atual

    .agg(

        F.count(
            "*"
        ).alias(
            "total_produtos"
        ),

        F.sum(
            "indicador_ruptura"
        ).alias(
            "produtos_em_ruptura"
        ),

        F.sum(
            "indicador_estoque_baixo"
        ).alias(
            "produtos_estoque_baixo"
        ),

        F.sum(
            "indicador_excesso"
        ).alias(
            "produtos_com_excesso"
        ),

        F.round(
            F.sum(
                "valor_estoque"
            ),
            2
        ).alias(
            "valor_total_estoque"
        ),

        F.round(
            F.sum(
                "valor_excedente"
            ),
            2
        ).alias(
            "capital_excedente"
        )
    )
)


display(
    df_kpis_estoque
)

In [0]:
display(

    df_analise_estoque_atual

    .groupBy(
        "classe_abc",
        "situacao_estoque"
    )

    .agg(
        F.count("*").alias(
            "quantidade_produtos"
        )
    )

    .orderBy(
        "classe_abc",
        "situacao_estoque"
    )
)

In [0]:
df_analise_estoque_atual = (

    df_analise_estoque_atual

    .withColumn(

        "score_prioridade",

        F.when(
            F.col(
                "prioridade_estoque"
            ) == "Crítica",
            5
        )

        .when(
            F.col(
                "prioridade_estoque"
            ) == "Muito alta",
            4
        )

        .when(
            F.col(
                "prioridade_estoque"
            ) == "Alta",
            3
        )

        .when(
            F.col(
                "prioridade_estoque"
            ).isin(
                [
                    "Revisar compras",
                    "Revisar portfólio"
                ]
            ),
            2
        )

        .otherwise(
            1
        )
    )
)

In [0]:
salvar_gold(
    df_analise_estoque_atual,
    "analise_estoque_atual"
)

In [0]:
display(

    df_analise_estoque_atual

    .filter(
        F.col(
            "score_prioridade"
        ) >= 3
    )

    .select(
        "score_prioridade",
        "classe_abc",
        "nome_produto",
        "categoria",
        "situacao_estoque",
        "dias_cobertura",
        "quantidade_estoque",
        "faturamento_12m",
        "acao_recomendada"
    )

    .orderBy(
        F.desc(
            "score_prioridade"
        ),
        F.asc(
            "ranking_faturamento"
        )
    )
)

In [0]:
valor_estoque_silver = (

    df_estoque_atual

    .agg(
        F.round(
            F.sum(
                "valor_estoque"
            ),
            2
        ).alias(
            "total"
        )
    )

    .first()[
        "total"
    ]
)


valor_estoque_gold = (

    df_analise_estoque_atual

    .agg(
        F.round(
            F.sum(
                "valor_estoque"
            ),
            2
        ).alias(
            "total"
        )
    )

    .first()[
        "total"
    ]
)


print(
    f"Silver: R$ {valor_estoque_silver:,.2f}"
)

print(
    f"Gold:   R$ {valor_estoque_gold:,.2f}"
)

In [0]:
diferenca = abs(
    float(valor_estoque_silver)
    -
    float(valor_estoque_gold)
)


if diferenca > 0.05:

    raise Exception(
        "O valor de estoque da Gold "
        "não corresponde à posição Silver."
    )


print(
    "Validação concluída: "
    "estoque Gold consistente com a Silver."
)

In [0]:
spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`analise_estoque_atual`
    IS 'Análise da posição atual de estoque combinada com Curva ABC, demanda, cobertura e prioridades operacionais.'
    """
)


spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`resumo_estoque`
    IS 'Resumo executivo das principais situações de estoque da Distribuidora Horizonte.'
    """
)


print(
    "Descrições adicionadas às tabelas."
)